In [2]:
import os
import torch
import pandas as pd
from PIL import Image
from IPython.display import display
from sentence_transformers import SentenceTransformer, util

from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F

In [ ]:
data_root = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset"

styles_path = os.path.join(data_root, "styles.csv")
image_dir   = os.path.join(data_root, "images")

df = pd.read_csv(styles_path, on_bad_lines="skip")

print(df.columns)  # to confirm id, productDisplayName, etc.
df.head()

In [ ]:
def image_path_from_id(pid):
    # Many fashion datasets name images as "<id>.jpg"
    path_jpg = os.path.join(image_dir, f"{pid}.jpg")
    path_png = os.path.join(image_dir, f"{pid}.png")

    if os.path.exists(path_jpg):
        return path_jpg
    if os.path.exists(path_png):
        return path_png
    return None

df["image_path"] = df["id"].apply(image_path_from_id)

# Keep only rows where the image exists
df = df[df["image_path"].notna()].reset_index(drop=True)

df[["id", "productDisplayName", "image_path"]].head()

In [ ]:
model_name = "sentence-transformers/clip-ViT-B-32"
model = SentenceTransformer(model_name)

In [ ]:
N = 2000  # or smaller/larger depending on speed
df_subset = df.head(N).copy()

images = [Image.open(p).convert("RGB") for p in df_subset["image_path"].tolist()]

image_embeddings = model.encode(
    images,
    batch_size=32,
    convert_to_tensor=True,
    show_progress_bar=True
)

image_embeddings.shape

In [ ]:
def search_images_by_text(query, top_k=5):
    # 1) Encode text query
    query_embedding = model.encode(
        query,
        convert_to_tensor=True
    )

    # 2) Cosine similarity with image embeddings
    cos_scores = util.cos_sim(query_embedding, image_embeddings)[0]

    # 3) Top-k indices
    top_results = torch.topk(cos_scores, k=top_k)

    results = []
    for score, idx in zip(top_results.values, top_results.indices):
        idx = idx.item()
        row = df_subset.iloc[idx]
        results.append({
            "row_index": idx,
            "id": row["id"],
            "name": row.get("productDisplayName", ""),
            "image_path": row["image_path"],
            "score": float(score)
        })
    return results

In [ ]:
import math
import matplotlib.pyplot as plt

def show_search_results_grid(results, thumb_size=(128, 128), rows=2, cols=5):
    n = len(results)
    if n == 0:
        print("No results.")
        return

    # If more results than slots, cut to rows*cols
    max_slots = rows * cols
    results = results[:max_slots]
    n = len(results)

    plt.figure(figsize=(cols * 2, rows * 2))

    for i, r in enumerate(results):
        img = Image.open(r["image_path"]).convert("RGB")
        img = img.resize(thumb_size)

        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(img)
        ax.set_title(str(r["id"]), fontsize=7)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
while True:
    q = input("Enter a text query (or 'exit'): ").strip()
    if q.lower() == "exit":
        break

    results = search_images_by_text(q, top_k=10)
    print("\nTop Matching Images:")
    show_search_results_grid(results, thumb_size=(128, 128), rows=2, cols=5)
    print("-" * 60)